# Implementation Notebook

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np

#### Config Decisions made during EDA

In [2]:
M_SMOOTHING = 25
MIN_CO_RATINGS = 5
MIN_MOVIE_RATES = 5
TOP_N_USER_RECS = 20
TOP_N_SIMILAR = 15
W_COLLAB = 0.75
W_CONTENT = 0.25
POS_RATING_FLOOR = 4.0
DB_PATH = "../backend/movies.db"

#### Load & Integrity

In [3]:
ratings = pd.read_csv("/home/bukunmi/code/givemore/data/ratings.csv")
movies = pd.read_csv("/home/bukunmi/code/givemore/data/movies.csv")
links = pd.read_csv("/home/bukunmi/code/givemore/data/links.csv")

In [4]:
assert set(ratings.columns) == {"userId", "movieId", "rating", "timestamp"}
assert set(movies.columns) == {"movieId", "title", "genres"}
assert set(links.columns) == {"movieId", "imdbId", "tmdbId"}

In [5]:
assert ratings["userId"].nunique() == 610
assert movies["movieId"].nunique() == 9742
assert links["movieId"].nunique() == 9742

In [6]:
assert set(ratings.movieId) - set(movies.movieId) == set()
assert set(movies.movieId) - set(links.movieId) == set()
assert ratings[["userId","movieId"]].duplicated().sum() == 0

#### Clean Movie Metadata

In [7]:
movies["raw_title"] = movies["title"].str.replace(r"\s+", " ", regex=True).str.strip()
movies["Year"] = movies["raw_title"].str.extract(r"\((\d{4})(?:[–-]\d{4})?\)\s*$", expand=False)
movies["title"] = movies["raw_title"].str.replace(r"\s*\(\d{4}(?:[–-]\d{4})?\)\s*$", "", regex=True)

In [8]:
movies.drop(columns=["raw_title"], inplace=True)

In [9]:
assert movies["Year"].isna().sum() == 12

In [10]:
movies["genre_list"] = movies["genres"].str.split("|")
movies["genre_list"] = movies["genre_list"].apply(lambda genres: [g for g in genres if g != "(no genres listed)"])

In [11]:
assert (movies["genres"] == "(no genres listed)").sum() == 34

#### Foundational Statistics

In [12]:
C = ratings["rating"].mean()
movie_stats = ratings.groupby("movieId")["rating"].agg(["count", "mean"])
user_stats = ratings.groupby("userId")["rating"].agg(["count", "mean"])

In [13]:
assert user_stats.shape[0] == 610
assert movie_stats.shape[0] == 9724
assert C == 3.501556983616962

#### Popular Fallback

In [14]:
def weighted_score(v, R, C, m):
    weighted_score = (v / (v + m)) * R + (m / (v + m)) * C
    return weighted_score

In [15]:
weighted_scores = movie_stats.apply(lambda row: weighted_score(row["count"], row["mean"], C, M_SMOOTHING), axis=1).sort_values(ascending=False)

In [16]:
weighted_scores.dtype

dtype('float64')

In [17]:
assert weighted_scores.shape[0] == 9724
assert weighted_scores.index[0] == 318 # Shawshank
assert np.isclose(weighted_scores.iloc[0], 4.361224925702994)

#### Rating matrices

In [27]:
R = ratings.pivot_table(index="userId", columns="movieId", values="rating")

In [29]:
movie_index = R.columns.to_numpy()
user_index  = R.index.to_numpy()

In [33]:
B = R.notna().astype(float)
co = (B.T @ B).to_numpy().copy()
np.fill_diagonal(co, 0)

In [34]:
Mc = R.sub(user_stats["mean"], axis=0).fillna(0.0).to_numpy()

In [44]:
assert (R/B).shape == (610, 9724)
assert co.shape == (9724, 9724)
assert Mc.shape == (610, 9724)
assert (co >= MIN_CO_RATINGS).sum() // 2 == 1_293_963

#### Collaborative similarity (adjusted cosine + IUF)